# Week 72, Day 5: Test architecture

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w072d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from datetime import date

def membership_status(expiry):
    days = (expiry - date.today()).days
    if days < 0:
        return "expired"
    if days <= 30:
        return "renew soon"
    return "active"

print(membership_status(date(2027, 1, 31)))
print(membership_status(date(2026, 11, 30)))

In [ ]:
from dataclasses import dataclass, field
from datetime import date

class FixedClock:
    def __init__(self, today):
        self._today = today
    def today(self):
        return self._today

@dataclass
class RecordingMailer:
    sent: list = field(default_factory=list)
    def send(self, to, subject):
        self.sent.append((to, subject))

def remind_expiring(members, clock, mailer, within=30):
    """The core: decisions only, with the clock and the mailer passed in."""
    reminded = []
    for email, expiry in members:
        if 0 <= (expiry - clock.today()).days <= within:
            mailer.send(email, "Your membership is ending soon")
            reminded.append(email)
    return reminded

members = [("ada@x", date(2026, 11, 20)), ("bob@x", date(2027, 3, 1)), ("cy@x", date(2026, 10, 1))]
mailer = RecordingMailer()
print(remind_expiring(members, FixedClock(date(2026, 11, 10)), mailer))
print(mailer.sent)

In [ ]:
class OrderBuilder:
    def __init__(self):
        self._items, self._member, self._code = [], False, None

    def with_item(self, name, price, qty=1):
        self._items.append({"name": name, "price": price, "qty": qty})
        return self

    def for_member(self):
        self._member = True
        return self

    def with_code(self, code):
        self._code = code
        return self

    def build(self):
        items = self._items or [{"name": "tea", "price": 3.0, "qty": 1}]
        return {"items": items, "member": self._member, "code": self._code}

def an_order():
    return OrderBuilder()

def total(order):
    subtotal = sum(i["price"] * i["qty"] for i in order["items"])
    return round(subtotal * (0.9 if order["member"] else 1), 2)

# arrange
order = an_order().with_item("jam", 4.0, qty=2).for_member().build()
# act
result = total(order)
# assert
assert result == 7.2
print(result, an_order().build())

> **Key idea.** Unit tests are fast and precise, integration tests check the joins, end-to-end tests check whole journeys; aim for many unit tests, fewer integration tests and a few end-to-end tests, testing each behaviour at the lowest level that can catch its bugs. Ports and adapters keep the core free of effects: inject the clock, configuration and services, wire real adapters in main and test ones in tests. Contract tests run one suite against every implementation, so fakes stay honest. Builders make test data with defaults; tests read arrange, act, assert. Flaky tests come from time, randomness, shared state, sleeping, networks and ordering: fix the cause, never retry. Lay tests out by kind, mark the slow ones, and name tests after behaviour.

### Exercise: Pyramid or cone?

Write `shape(counts)`: `counts` is a dict with `"unit"`, `"integration"` and `"e2e"` numbers of tests. Return `"pyramid"` if unit > integration > e2e; `"ice-cream cone"` if e2e > integration > unit; `"hourglass"` if both unit and e2e are greater than integration; otherwise `"unclear"`. Then `suite_time(counts, seconds)`: the total running time, given average `seconds` per test of each kind (a dict with the same keys), and `fast_feedback(counts, seconds, budget)`: the kinds that can run within `budget` seconds, taken in the order unit, integration, e2e and stopping at the first that would go over.

In [ ]:
def shape(counts):
    ___

def suite_time(counts, seconds):
    ___

def fast_feedback(counts, seconds, budget):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shape():\n    \"shapes and budgets\"\n    assert shape({\"unit\": 500, \"integration\": 80, \"e2e\": 10}) == \"pyramid\"\n    assert shape({\"unit\": 20, \"integration\": 60, \"e2e\": 300}) == \"ice-cream cone\"\n    assert shape({\"unit\": 400, \"integration\": 20, \"e2e\": 200}) == \"hourglass\"\n    assert shape({\"unit\": 50, \"integration\": 50, \"e2e\": 50}) == \"unclear\"\n    secs = {\"unit\": 0.01, \"integration\": 0.5, \"e2e\": 6}\n    counts = {\"unit\": 500, \"integration\": 80, \"e2e\": 10}\n    assert abs(suite_time(counts, secs) - 105) < 1e-9\n    assert fast_feedback(counts, secs, 60) == [\"unit\", \"integration\"] and fast_feedback(counts, secs, 2) == []\n    assert fast_feedback(counts, secs, 200) == [\"unit\", \"integration\", \"e2e\"]")

### Exercise: A test data builder

Write `UserBuilder`, a builder for user dicts with the keys `"name"` (default `"Ada"`), `"email"` (default: the lower-cased name with `"@example.com"`, worked out at `build()` time, so it follows a changed name), `"roles"` (default an empty list) and `"active"` (default `True`). Methods, each returning the builder: `named(name)`, `with_email(email)`, `with_role(role)` (adding one), `inactive()`. `build()` returns a **new** dict each time, so changing one built user never affects another. Also `a_user()`, returning a new builder.

In [ ]:
class UserBuilder:
    ___

def a_user():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_builder():\n    \"defaults and overrides\"\n    assert a_user().build() == {\"name\": \"Ada\", \"email\": \"ada@example.com\", \"roles\": [], \"active\": True}\n    bob = a_user().named(\"Bob\").with_role(\"admin\").with_role(\"editor\").inactive().build()\n    assert bob == {\"name\": \"Bob\", \"email\": \"bob@example.com\", \"roles\": [\"admin\", \"editor\"], \"active\": False}\n    assert a_user().named(\"Cy\").with_email(\"c@x\").build()[\"email\"] == \"c@x\"\n    b = a_user().with_role(\"viewer\")\n    one, two = b.build(), b.build()\n    one[\"roles\"].append(\"hacked\")\n    assert two[\"roles\"] == [\"viewer\"] and b.build()[\"roles\"] == [\"viewer\"]")

### Exercise: Injecting the outside world

The function in the starter reads the date, the environment and a file all by itself, so it can't be tested reliably. Rewrite it as `daily_message(today, config, read_quote)`: `today` is a `date`, `config` a dict (it reads `"NAME"`, defaulting to `"friend"`, and `"SHOUTY"`, where only `"1"` means yes), and `read_quote(n)` a function returning quote number `n`. The message is `f"Good {part}, {name}! Quote {n}: {quote}"`, where `part` is `"weekend"` on Saturday and Sunday and `"weekday"` otherwise, `n` is the day of the year (`today.timetuple().tm_yday`), and the whole message is upper-cased if shouty. Then `main()`, which wires in the real date, `os.environ` and a real file reader (it isn't called by the checks).

In [ ]:
import os
from datetime import date

def daily_message():
    today = date.today()
    name = os.environ.get("NAME", "friend")
    n = today.timetuple().tm_yday
    with open("quotes.txt") as f:
        quote = f.read().splitlines()[n % 100]
    part = "weekend" if today.weekday() >= 5 else "weekday"
    message = f"Good {part}, {name}! Quote {n}: {quote}"
    return message.upper() if os.environ.get("SHOUTY") == "1" else message

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_injected():\n    \"everything the function needs is passed in\"\n    from datetime import date\n    asked = []\n    def fake_quote(n):\n        asked.append(n)\n        return \"Simple is better than complex.\"\n    msg = daily_message(date(2026, 11, 14), {\"NAME\": \"Ada\"}, fake_quote)\n    assert msg == \"Good weekend, Ada! Quote 318: Simple is better than complex.\" and asked == [318]\n    loud = daily_message(date(2026, 11, 10), {\"SHOUTY\": \"1\"}, lambda n: \"Flat is better.\")\n    assert loud == \"GOOD WEEKDAY, FRIEND! QUOTE 314: FLAT IS BETTER.\"\n    assert daily_message(date(2026, 11, 10), {\"SHOUTY\": \"yes\"}, lambda n: \"x\").startswith(\"Good\")\n    body = source.split(\"def daily_message\")[1].split(\"\\ndef \")[0]\n    assert \"date.today\" not in body and \"environ\" not in body, \"daily_message still reads the date or the environment itself\"")

### Exercise: One contract, two implementations

`repos.py` has two implementations of a key-value repository port: `MemoryRepo()` and `FileRepo(path)` (storing JSON in a file). Both must: `add(key, value)` (raising `KeyError` if the key exists), `get(key)` (raising `KeyError` if it doesn't), `remove(key)` (raising `KeyError` if it doesn't exist) and `keys()` (a sorted list). Write **contract tests**: a fixture parametrized over both (use `tmp_path` for the file), and tests that pass on the correct code and catch a broken version of either.

In [ ]:
# Run this cell first: it creates the files used below.
with open('repos.py', 'w', encoding='utf-8') as f:
    f.write('import json\n\nclass MemoryRepo:\n    def __init__(self):\n        self._data = {}\n\n    def add(self, key, value):\n        if key in self._data:\n            raise KeyError(key)\n        self._data[key] = value\n\n    def get(self, key):\n        return self._data[key]\n\n    def remove(self, key):\n        del self._data[key]\n\n    def keys(self):\n        return sorted(self._data)\n\n\nclass FileRepo:\n    def __init__(self, path):\n        self.path = path\n        if not path.exists():\n            path.write_text("{}")\n\n    def _load(self):\n        return json.loads(self.path.read_text())\n\n    def _save(self, data):\n        self.path.write_text(json.dumps(data))\n\n    def add(self, key, value):\n        data = self._load()\n        if key in data:\n            raise KeyError(key)\n        data[key] = value\n        self._save(data)\n\n    def get(self, key):\n        return self._load()[key]\n\n    def remove(self, key):\n        data = self._load()\n        del data[key]\n        self._save(data)\n\n    def keys(self):\n        return sorted(self._load())\n')

In [ ]:
import pytest
from repos import MemoryRepo, FileRepo

@pytest.fixture
def repo():
    return MemoryRepo()

def test_add_and_get(repo):
    repo.add("a", 1)
    assert repo.get("a") == 1

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"repos.py\").read()\nBROKEN = {\n    \"memory overwrites duplicates\": GOOD.replace(\"        if key in self._data:\\n            raise KeyError(key)\\n\", \"\"),\n    \"file overwrites duplicates\": GOOD.replace(\"        if key in data:\\n            raise KeyError(key)\\n\", \"\"),\n    \"file remove doesn't save\": GOOD.replace(\"        del data[key]\\n        self._save(data)\", \"        del data[key]\"),\n    \"file keys unsorted\": GOOD.replace(\"return sorted(self._load())\", \"return list(self._load())\"),\n    \"memory remove of a missing key passes\": GOOD.replace(\"        del self._data[key]\", \"        self._data.pop(key, None)\"),\n}\ndef test_contract():\n    \"one suite catches a bug in either implementation\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and len(r[\"passed\"]) >= 8, r[\"output\"][-600:]\n    results = against(source, \"repos\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"repos.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: An integration test with real files

`notes_app.py` has two modules' worth of code working together: `NoteStore(folder)` saves each note as a file in a folder, and `search(store, word)` finds notes containing a word. Write **integration** tests using real files in `tmp_path` (no mocks), checking what ends up on disk as well as what the functions return, that pass on the correct code and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('notes_app.py', 'w', encoding='utf-8') as f:
    f.write('from pathlib import Path\n\nclass NoteStore:\n    def __init__(self, folder):\n        self.folder = Path(folder)\n        self.folder.mkdir(parents=True, exist_ok=True)\n\n    def save(self, title, text):\n        path = self.folder / f"{title}.txt"\n        path.write_text(text, encoding="utf-8")\n        return path\n\n    def titles(self):\n        return sorted(p.stem for p in self.folder.glob("*.txt"))\n\n    def read(self, title):\n        return (self.folder / f"{title}.txt").read_text(encoding="utf-8")\n\ndef search(store, word):\n    word = word.lower()\n    return [t for t in store.titles() if word in store.read(t).lower()]\n')

In [ ]:
from notes_app import NoteStore, search

def test_save(tmp_path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"notes_app.py\").read()\nBROKEN = {\n    \"saves without the extension\": GOOD.replace('f\"{title}.txt\"', 'f\"{title}\"', 1),\n    \"doesn't create nested folders\": GOOD.replace(\"parents=True, exist_ok=True\", \"exist_ok=True\"),\n    \"case-sensitive search\": GOOD.replace(\"    word = word.lower()\\n\", \"\").replace(\"store.read(t).lower()\", \"store.read(t)\"),\n    \"titles unsorted\": GOOD.replace(\"return sorted(p.stem\", \"return list(p.stem\"),\n    \"search looks at titles only\": GOOD.replace(\"if word in store.read(t).lower()\", \"if word in t.lower()\"),\n}\ndef test_catches():\n    \"real files catch all five broken versions\"\n    results = against(source, \"notes_app\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"notes_app.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Bug hunt: A test that broke on its own

`membership_status` reads today's date itself, so this test passed when it was written and fails now. Give `membership_status` a `today` parameter and pass a fixed date in the test, so the test passes on any day.

In [ ]:
from datetime import date

def membership_status(expiry):
    days = (expiry - date.today()).days
    if days < 0:
        return "expired"
    if days <= 30:
        return "renew soon"
    return "active"

def test_renew_soon():
    assert membership_status(date(2025, 6, 20)) == "renew soon"

def test_active():
    assert membership_status(date(2025, 12, 31)) == "active"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"the date is injected\"\n    r = pytest_run(source)\n    assert sorted(r[\"passed\"]) == [\"test_active\", \"test_renew_soon\"], r[\"output\"][-600:]\n    assert \"date.today()\" not in source.split(\"def test_\")[0].split(\"def membership_status\")[1], \"membership_status still reads the date itself\"")

### Bug hunt: State left behind

`seen` is a module-level cache, and nothing resets it, so `test_first_visit` passes only if it runs before `test_second_visit` changes it. Add an **autouse** fixture that clears `seen` before each test, so the tests pass in any order.

In [ ]:
import pytest

seen = set()

def visit(page):
    first = page not in seen
    seen.add(page)
    return "welcome" if first else "welcome back"

def test_second_visit():
    visit("home")
    assert visit("home") == "welcome back"

def test_first_visit():
    assert visit("home") == "welcome"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"independent of order\"\n    r = pytest_run(source)\n    assert sorted(r[\"passed\"]) == [\"test_first_visit\", \"test_second_visit\"], r[\"output\"][-600:]\n    swapped = source.replace(\"def test_first_visit\", \"def test_aaa_first_visit\")\n    assert pytest_run(swapped)[\"failed\"] == []\n    assert \"autouse\" in source")

### Bug hunt: Unseeded randomness

`shuffle_deck` uses the global random generator, so the test's expected first card is only right by luck: here it's wrong. Make `shuffle_deck` take a `random.Random` to use, and pass `random.Random(7)`, so this prints `True` every time.

In [ ]:
import random

def shuffle_deck(cards):
    deck = list(cards)
    random.shuffle(deck)
    return deck

deck = shuffle_deck(range(10))
print(deck == shuffle_deck(range(10)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"reproducible\"\n    expect_output(\"True\")\n    assert \"Random(\" in source")

### Puzzle: A contract-test fixture

Arrange the lines of a fixture that gives each test an in-memory repository and then a file-based one. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the shape with many end-to-end tests and few unit tests: "pyramid" or "ice-cream cone"
q = ___   # the three parts of a well-laid-out test, joined with ", "
r = ___   # True or False: adding a retry is a good fix for a flaky test
s = ___   # what a contract test is run against: "the fake only" or "every implementation"
t = ___   # in ports and adapters, the part that wires the real adapters together: "main" or "the core"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"ice-cream cone\" and q == \"arrange, act, assert\" and r is False, \"p, q and r\"\n    assert s == \"every implementation\" and t == \"main\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: choosing which tests to run

Large projects don't run every test on every change. Write `select_tests(changed, imports, covers, history)`: `changed` is a set of module names that changed; `imports` maps each module to the set of modules it imports; `covers` maps each test name to the set of modules it ran (from coverage); `history` maps a test name to `(failed_last_time, seconds)`. A module is **affected** if it changed or imports, directly or indirectly, an affected module. Select every test that covers an affected module, and return them ordered: tests that failed last time first, then by `seconds` (fastest first), then by name. A test missing from `history` counts as `(True, 0)` (new tests run early). Also `affected_modules(changed, imports)`, returning the sorted list.

In [ ]:
def affected_modules(changed, imports):
    ___

def select_tests(changed, imports, covers, history):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_selection():\n    \"affected modules and test order\"\n    imports = {\"app\": {\"orders\", \"users\"}, \"orders\": {\"pricing\", \"db\"}, \"users\": {\"db\"}, \"pricing\": set(), \"db\": set(), \"report\": {\"app\"}}\n    assert affected_modules({\"pricing\"}, imports) == [\"app\", \"orders\", \"pricing\", \"report\"]\n    assert affected_modules({\"db\"}, imports) == [\"app\", \"db\", \"orders\", \"report\", \"users\"]\n    covers = {\n        \"test_price\": {\"pricing\"},\n        \"test_orders\": {\"orders\", \"pricing\", \"db\"},\n        \"test_users\": {\"users\", \"db\"},\n        \"test_report\": {\"report\", \"app\", \"orders\", \"users\", \"pricing\", \"db\"},\n        \"test_new\": {\"pricing\"},\n    }\n    history = {\"test_price\": (False, 0.1), \"test_orders\": (False, 2.0), \"test_users\": (True, 0.5), \"test_report\": (True, 9.0)}\n    assert select_tests({\"pricing\"}, imports, covers, history) == [\"test_new\", \"test_report\", \"test_price\", \"test_orders\"]\n    assert select_tests({\"users\"}, imports, covers, history) == [\"test_users\", \"test_report\"]\n    assert select_tests(set(), imports, covers, history) == []")

## Warm-up

What this prints depends on the day you run it. A test written with these dates passed on the day it was written, and breaks, by itself, some weeks later: no code changed, nobody did anything wrong, and the build is red. The fix isn't in the test. It's in the code's **shape**: `membership_status` reaches out for the date instead of being given it.

## Kinds of test, and the pyramid

Tests differ in how much of the system they exercise:

- **Unit tests** check one small piece (a function, a class) in isolation, with its slow or external collaborators replaced. They run in milliseconds, point straight at the broken code, and can test every edge case cheaply.
- **Integration tests** check that pieces work **together** with real collaborators: your code with a real database, two modules through their real interface, a request handler with its real routing. They catch the bugs that live in the joins, which unit tests can't see, but they're slower and a failure is harder to locate.
- **End-to-end (system) tests** drive the whole application as a user would: start the server, send real HTTP requests, check the responses. They give the most confidence per test and the least information per failure, and they're the slowest and flakiest.

The **test pyramid** is the usual advice on proportions: **many** unit tests at the bottom, **fewer** integration tests, and a **handful** of end-to-end tests at the top, covering the most important journeys. The opposite shape, the **ice-cream cone** (mostly manual and end-to-end tests, few unit tests), is slow and brittle. Some teams argue for a **trophy** shape with integration tests as the widest band, because modern tools make them fast. The principle behind all three is the same: test each behaviour at the **lowest level that can catch its bugs**, and use the higher levels for what only they can check.

"Unit" doesn't have to mean "one function with every collaborator mocked". Testing a small cluster of real classes together, with doubles only at the edges (Day 2), usually gives tests that survive refactoring far better.

## Designing for testability

Code is easy to test when its **decisions** are separated from its **effects**. **Ports and adapters** (also called **hexagonal architecture**) makes that a design rule:

- The **core** holds the logic, as plain functions and classes. It never reads the clock, the environment, files or the network itself.
- It talks to the outside world through **ports**: small interfaces it defines, such as `Clock.now()`, `Repository.save(order)` or `Mailer.send(to, subject)`.
- **Adapters** implement the ports for real (a `SystemClock`, a `SqliteRepository`, an `SmtpMailer`) or for tests (a `FixedClock`, an `InMemoryRepository`, a `RecordingMailer`).
- The program's entry point (its `main`) **wires** the real adapters together. Tests wire the test ones. This is **dependency injection**: objects receive their collaborators instead of creating them.

The payoff: the core is tested with fast unit tests and simple fakes; each adapter gets a few integration tests against the real thing; and the wiring gets one or two end-to-end tests.

The same `remind_expiring` runs in production with a clock that reads the real date and a mailer that really sends; nothing in it changes.

## Contract tests and test data

**Fakes drift.** An `InMemoryRepository` written to match a real database can slowly stop matching it: the real one raises on a duplicate key and the fake silently overwrites, and every test using the fake now tests a fiction. The defence is a **contract test**: one suite describing what **any** implementation of the port must do, run against **every** implementation, the real adapter included. With pytest that's a parametrized fixture (Day 1): `@pytest.fixture(params=["memory", "file"])` that builds each implementation in turn.

**Test data.** Tests need objects, and building them inline buries the one value that matters among a dozen that don't. A **test data builder** provides sensible defaults and lets each test state only what it cares about: `an_order().with_item("tea", 3.50, qty=2).for_member().build()`. When the `Order` class gains a field, one builder changes, not three hundred tests.

**Arrange, act, assert.** Lay out each test in three parts, separated by a blank line: set up the situation, do the one thing being tested, check the result. One act per test; if the name of a test needs "and", it's probably two tests.

## Flaky tests, and a suite's layout

A **flaky** test sometimes passes and sometimes fails with no code change. Flaky tests are worse than no tests: people learn to re-run until green, and then ignore a real failure. The usual causes, and their cures:

| Cause | Cure |
|---|---|
| The current time or date | inject a clock |
| Randomness | inject a seeded `random.Random` |
| Test order and shared state (globals, caches, module fixtures that tests change) | fresh state per test; an autouse fixture that resets globals |
| Waiting a fixed time for something asynchronous (`sleep(1)` and hope) | wait for the condition with a timeout, or make the work synchronous in tests |
| The network or another real service | a fake, or a local container that the test controls |
| Iteration order of sets, or floating-point details | sort before comparing; `pytest.approx` |

When you find one, don't add a retry: find the cause. Run the suite in random order (the `pytest-randomly` plugin) and repeatedly (`pytest-repeat`) to flush out order and timing problems.

**Layout.** Keep tests in a `tests/` folder mirroring the package (`tests/unit/`, `tests/integration/`, each with a `conftest.py`). Mark the slow ones (`@pytest.mark.slow`, registered in the project's configuration) so that developers can run `pytest -m "not slow"` every few minutes, and the full suite runs in CI (Week 74). Name tests for the behaviour they check (`test_expired_membership_is_not_renewed`), so the list of failures reads like a list of broken promises.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the tests you've written in this course and sort a dozen of them into unit, integration and end-to-end. Tomorrow's Forge puts the whole week together: a test suite, from fixtures to properties to mutation testing, for a small booking system.